# Model Training & Hyperparameter Tuning
**COEN 330 - Applied Machine Learning**

Trains and tunes **five** classifiers on the loan-approval task using **5-fold StratifiedKFold
cross-validation on the training set only**. The test set is never touched here.

**Leakage-safe design:** the preprocessor is placed *inside* a scikit-learn `Pipeline`, so
the scaler/encoder refit on each fold's training portion during CV (not on the whole
training set beforehand). Each saved model is therefore an **end-to-end pipeline**
(raw DataFrame in → prediction out).

**Positive class = approved (loan_status = 1), the minority (~22%)**, so we use
`class_weight='balanced'` where supported and select on **PR-AUC**.

**Reported metrics:** accuracy, balanced accuracy, precision, recall, F1, PR-AUC.
PR-AUC is the **selection** metric (threshold-independent, focused on the minority class);
the others are reported alongside.

Models:
1. Logistic Regression: baseline model
2. SVM (RBF kernel): kernel / margin-based
3. Gaussian Naive Bayes: probabilistic
4. Random Forest: bagging ensemble
5. Gradient Boosting (HistGradientBoosting): boosting ensemble

A `DummyClassifier` is included as a sanity-check, not one of the five models.

## Setup

In [ ]:
import sys, json, time
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src'))   # make src/ importable

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from preprocessing import load_data, split_data, build_preprocessor, get_feature_names
from utils import DATA_RAW, DATASET_FILE, MODELS_DIR, RESULTS_DIR, SEED

DATA_PATH = DATA_RAW / DATASET_FILE

# 5-fold stratified CV, fixed seed for reproducibility.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Multi-metric scoring
SCORING = {
    'pr_auc':       'average_precision',    # primary: PR-AUC on the minority positive class
    'accuracy':     'accuracy',
    'balanced_acc': 'balanced_accuracy',
    'precision':    'precision',            # precision on approved (pos_label=1)
    'recall':       'recall',               # recall on approved (pos_label=1)
    'f1':           'f1',
}
PRIMARY = 'pr_auc'

## 1. Load & split: Primary feature set (WITH `previous_loan_defaults_on_file`)

`X_train` / `X_test` stay as **raw DataFrames**; the preprocessor runs inside the pipeline
during CV. The split is deterministic (fixed seed), so `04_evaluation` reproduces the
identical split by calling `split_data` again.

In [ ]:
df = load_data(DATA_PATH, add_engineered=False)
X_train, X_test, y_train, y_test = split_data(df)

print(f'Train rows: {len(X_train)}, Test rows: {len(X_test)}')
print(f'Approved (1) share — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}')

## 2. Models and hyperparameter grids

In [ ]:
def build_models():
    """Return {name: (estimator, param_grid)}. Grid keys are bare param names;
    run_search() prefixes them with 'clf__' for the pipeline. Grids kept modest for
    runtime."""
    return {
        # Interpretable baseline. saga + l1_ratio gives elastic-net (L1<->L2 sweep).
        'LogisticRegression': (
            LogisticRegression(solver='saga', class_weight='balanced',
                               max_iter=2000, random_state=SEED),
            {'C': [0.01, 0.1, 1, 10], 'l1_ratio': [0, 1]},
        ),
        # SVM (RBF): slowest on this many rows; small grid. Uses decision_function for
        # PR-AUC (no probability=True needed -> faster). class_weight handles imbalance.
        'SVM_RBF': (
            SVC(kernel='rbf', class_weight='balanced', random_state=SEED),
            {'C': [1, 10], 'gamma': ['scale']},
        ),
        # GaussianNB: independence assumption broken by one-hot + derived cols, so expect
        # it to trail -> a clean error-analysis talking point. Barely tunes.
        'GaussianNB': (
            GaussianNB(),
            {'var_smoothing': [1e-9, 1e-8, 1e-7]},
        ),
        'RandomForest': (
            RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1),
            {'n_estimators': [200, 400], 'max_depth': [None, 10, 20]},
        ),
        # HistGradientBoosting: fast boosting, no external dependency. No class_weight ->
        # imbalance handled via threshold tuning in 04 (and PR-AUC model selection here).
        'GradientBoosting': (
            HistGradientBoostingClassifier(random_state=SEED),
            {'learning_rate': [0.05, 0.1], 'max_depth': [None, 6],
             'max_iter': [200, 400]},
        ),
    }

print('Models:', list(build_models().keys()))

## 3. Pipeline + Tuning Routine

`make_pipeline_for` wraps the preprocessor and a classifier so the preprocessor refits
**inside each CV fold** (leakage-safe). `run_search` tunes each pipeline with `GridSearchCV`,
refit on PR-AUC, and records every metric's mean CV score at the best setting.

In [ ]:
def make_pipeline_for(estimator, drop_prev_defaults=False, add_engineered=False):
    return Pipeline([
        ('pre', build_preprocessor(drop_prev_defaults=drop_prev_defaults,
                                   add_engineered=add_engineered)),
        ('clf', estimator),
    ])

def run_search(X, y, models, drop_prev_defaults=False, add_engineered=False):
    rows, best = [], {}
    for name, (est, grid) in models.items():
        t0 = time.time()
        pipe = make_pipeline_for(est, drop_prev_defaults, add_engineered)
        pgrid = {f'clf__{k}': v for k, v in grid.items()}   # target the classifier step
        gs = GridSearchCV(pipe, pgrid, scoring=SCORING, refit=PRIMARY,
                          cv=cv, n_jobs=-1, error_score='raise')
        gs.fit(X, y)                                          # X is a RAW DataFrame
        i = gs.best_index_
        rows.append({
            'model': name,
            'pr_auc':       gs.cv_results_['mean_test_pr_auc'][i],
            'accuracy':     gs.cv_results_['mean_test_accuracy'][i],
            'balanced_acc': gs.cv_results_['mean_test_balanced_acc'][i],
            'precision':    gs.cv_results_['mean_test_precision'][i],
            'recall':       gs.cv_results_['mean_test_recall'][i],
            'f1':           gs.cv_results_['mean_test_f1'][i],
            'best_params':  {k.replace('clf__', ''): v for k, v in gs.best_params_.items()},
            'fit_sec':      round(time.time() - t0, 1),
        })
        best[name] = gs.best_estimator_                       # full fitted pipeline
        print(f'{name:18s} PR-AUC={rows[-1]["pr_auc"]:.4f}  F1={rows[-1]["f1"]:.4f}  '
              f'P={rows[-1]["precision"]:.4f}  R={rows[-1]["recall"]:.4f}  '
              f'({rows[-1]["fit_sec"]}s)  {rows[-1]["best_params"]}')
    res = (pd.DataFrame(rows).sort_values('pr_auc', ascending=False).reset_index(drop=True))
    return res, best

## 4. Dummy baseline (sanity-check)

Predict the majority class (rejected). Works directly on the raw DataFrame (it ignores
features). High accuracy here means we should not select on accuracy.

In [ ]:
from sklearn.metrics import average_precision_score, recall_score
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
dummy_proba = dummy.predict_proba(X_test)[:, 1]
print(f'Dummy (most_frequent): test accuracy = {dummy.score(X_test, y_test):.3f}')
print(f'  PR-AUC (test) = {average_precision_score(y_test, dummy_proba):.3f}  '
      f'(≈ base rate {y_test.mean():.3f})')
print(f'  Recall on approved = {recall_score(y_test, dummy.predict(X_test)):.3f}  '
      f'(predicts no approvals)')

## 5. Run tuning on the primary feature set

In [ ]:
models = build_models()
results_primary, best_models = run_search(X_train, y_train, models,
                                          drop_prev_defaults=False, add_engineered=False)

print('\nCV results (sorted by PR-AUC):')
display(results_primary.drop(columns='best_params').round(4))

## 6. Save tuned pipelines and CV results

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Each saved model is a full pipeline (preprocessor + classifier) -> 04 loads it and
# predicts directly on the raw X_test DataFrame.
for name, pipe in best_models.items():
    joblib.dump(pipe, MODELS_DIR / f'{name}.pkl')

# Feature names from any fitted pipeline's preprocessor (for interpretation in 04).
fitted_pre = best_models[next(iter(best_models))].named_steps['pre']
feature_names = get_feature_names(fitted_pre)
with open(MODELS_DIR / 'feature_names.json', 'w') as f:
    json.dump(feature_names, f)

results_primary.to_csv(RESULTS_DIR / 'cv_results_primary.csv', index=False)
print(f'Saved {len(best_models)} tuned pipelines to {MODELS_DIR}')
print(f'{len(feature_names)} encoded features recorded')
print(f'Saved CV results to {RESULTS_DIR / "cv_results_primary.csv"}')

## 7. (Optional) Feature-set experiments 

Re-run the same tuning on alternative feature sets and compare:
- **WITHOUT** `previous_loan_defaults_on_file`: how much does the near-perfect shortcut carry?
- **WITH engineered ratios**: do `employment_experience_ratio` / `credit_history_ratio` help?

The preprocessor flags are passed straight into `run_search`, so the same leakage-safe
pipeline is used for every arm. Uncomment to run (slow: re-tunes every model per arm).

In [ ]:
def run_feature_set(drop_prev_defaults, add_engineered, tag):
    df_ = load_data(DATA_PATH, add_engineered=add_engineered)
    Xtr, _, ytr, _ = split_data(df_)
    res, _ = run_search(Xtr, ytr, build_models(),
                        drop_prev_defaults=drop_prev_defaults, add_engineered=add_engineered)
    res.insert(0, 'feature_set', tag)
    return res

# --- uncomment to run the experiments ---
# arms = [
#     run_feature_set(False, False, 'WITH_prevdef'),
#     run_feature_set(True,  False, 'WITHOUT_prevdef'),
#     run_feature_set(False, True,  'WITH_engineered'),
# ]
# comparison = pd.concat(arms, ignore_index=True)
# comparison.to_csv(RESULTS_DIR / 'feature_set_comparison.csv', index=False)
# display(comparison.pivot_table(index='model', columns='feature_set', values='pr_auc').round(4))